In [3]:
!pip install -q transformers accelerate faiss-cpu

from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

In [4]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

MODEL_ID = "google/gemma-2-2b"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID, torch_dtype=torch.float16).cuda().eval()

def embed(texts: list[str]) -> np.ndarray:
    batch = tok(texts, return_tensors="pt", padding=True,
                truncation=True, max_length=256).to("cuda")
    with torch.no_grad():
        hidden = model(**batch).last_hidden_state        # (B, T, D)
    mask = batch["attention_mask"].unsqueeze(-1)          # (B, T, 1)
    pooled = (hidden * mask).sum(1) / mask.sum(1)         # mean pooling
    vec = torch.nn.functional.normalize(pooled.float(), dim=-1)
    return vec.cpu().numpy()

print(embed(["hello world"]).shape)   # (1, 2304)

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
!pip install -q transformers accelerate faiss-cpu


In [ ]:
import os
os.makedirs("app", exist_ok=True)
open("app/__init__.py", "w").close()

In [ ]:
%%writefile app/detection.py
import re

RULES = {
    "CREDIT_CARD": re.compile(r"\b(?:\d[ -]?){12,18}\d\b"),
    "EMAIL": re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+"),
    "SSN":   re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
    "PHONE": re.compile(r"(?<!\d)\+?\d[\d\s-]{8,}\d(?!\d)"),
    "IPV4":  re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b"),
}

def luhn_valid(number: str) -> bool:
    digits = [int(c) for c in number if c.isdigit()]
    if not 13 <= len(digits) <= 19:
        return False
    total = 0
    for i, d in enumerate(reversed(digits)):
        if i % 2 == 1:
            d *= 2
            if d > 9:
                d -= 9
        total += d
    return total % 10 == 0

def detect(text: str) -> list[tuple[str, int, int]]:
    out = []
    for label, pat in RULES.items():
        for m in pat.finditer(text):
            if label == "CREDIT_CARD" and not luhn_valid(m.group(0)):
                continue
            out.append((label, m.start(), m.end()))

    out.sort(key=lambda x: (x[1], -(x[2] - x[1])))
    result, last_end = [], -1
    for f in out:
        if f[1] >= last_end:
            result.append(f)
            last_end = f[2]
    return result

def sanitize(text: str) -> str:
    for label, s, e in sorted(detect(text), key=lambda x: x[1], reverse=True):
        text = text[:s] + f"<{label}>" + text[e:]
    return text

In [ ]:
from app.detection import detect, sanitize

sample = "Card 4111 1111 1111 1111 se payment, email a@b.com, SSN 123-45-6789"
print(detect(sample))
print(sanitize(sample))

In [ ]:
import faiss
from app.detection import sanitize

sentences = [
    "Mera email jane@example.com hai",
    "Mujhe apna phone number 98765 43210 par call karo",
    "Card 4111 1111 1111 1111 se payment hua",
    "Aaj mausam bahut achha hai",
    "Main cricket khelna pasand karta hoon",
]

# Pehle sanitize, phir embed (privacy rule)
sanitized = [sanitize(s) for s in sentences]
vecs = embed(sanitized)

index = faiss.IndexFlatIP(vecs.shape[1])
index.add(vecs)

query = "payment ka record kaise dekhun"
q = embed([sanitize(query)])
scores, ids = index.search(q, 3)

for i, s in zip(ids[0], scores[0]):
    print(round(float(s), 3), sanitized[i])

In [ ]:
queries = [
    ("kisi ka phone number kya hai", 1),
    ("email bhejna hai", 0),
    ("credit card se paisa diya", 2),
    ("kal ka weather kaisa rahega", 3),
    ("cricket ka match", 4),
]

hits = 0
for q, expected in queries:
    qv = embed([sanitize(q)])
    scores, ids = index.search(qv, 3)
    top = list(ids[0])
    ok = expected in top
    hits += ok
    print("✅" if ok else "❌", q, "->", sanitized[top[0]])

print("Top-3 accuracy:", hits / len(queries))

In [ ]:
for q, expected in queries:
    qv = embed([sanitize(q)])
    scores, ids = index.search(qv, 3)
    print("\nQuery:", q)
    for i, s in zip(ids[0], scores[0]):
        print("  ", round(float(s), 3), sanitized[i])

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def embed_st(texts):
    return st_model.encode(texts, normalize_embeddings=True)

vecs2 = embed_st(sanitized)
index2 = faiss.IndexFlatIP(vecs2.shape[1])
index2.add(vecs2)

hits2 = 0
for q, expected in queries:
    qv = embed_st([sanitize(q)])
    scores, ids = index2.search(qv, 3)
    top = list(ids[0])
    ok = expected in top
    hits2 += ok
    print("✅" if ok else "❌", q, "->", sanitized[top[0]], round(float(scores[0][0]), 3))

print("MiniLM Top-3 accuracy:", hits2 / len(queries))

In [ ]:
!ls -R app

In [ ]:
%cd /content/gemma-embeddings-practice
!git pull origin main

In [ ]:
%cd /content
!rm -rf gemma-embeddings-practice
!git clone https://github.com/YOUR_USERNAME/gemma-embeddings-practice.git

In [ ]:
import os
from google.colab import userdata
from google.colab.userdata import SecretNotFoundError, NotebookAccessError

try:
    # Trying to get GITHUB_TOKEN from Colab Secrets
    GH_TOKEN = userdata.get("GITHUB_TOKEN")
    USER = "Muskan-Savita"
    REPO = "gemma-embeddings-practice"

    %cd /content
    !rm -rf gemma-embeddings-practice
    !git clone https://{GH_TOKEN}@github.com/{USER}/{REPO}.git
except NotebookAccessError:
    print("❌ Access Error: 'GITHUB_TOKEN' exists but this notebook does not have access to it.")
    print("Please go to the Secrets panel (🔑 key icon on the left sidebar) and turn ON the toggle switch for 'GITHUB_TOKEN' to grant Notebook access.")
except SecretNotFoundError:
    print("❌ Error: 'GITHUB_TOKEN' was not found in your Colab Secrets.")
    print("Please add 'GITHUB_TOKEN' to your Secrets panel (🔑 key icon on the left sidebar) and enable Notebook access.")


In [ ]:
%cd /content/gemma-embeddings-practice
!cp -r /content/app .
!git status

In [ ]:
!git add app
!git commit -m "Add PII detection module"
!git push origin main

In [5]:
!git config --global user.email "savitamuskan1404@gmail.com"
!git config --global user.name "Muskan-Savita"
!git commit -m "Add PII detection module"
!git push origin main

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git


In [6]:
%cd /content/gemma-embeddings-practice
!git config --global user.email "savitamuskan1404@gmail.com"
!git config --global user.name "Muskan-Savita"
!git add app
!git commit -m "Add PII detection module"
!git push origin main

[Errno 2] No such file or directory: '/content/gemma-embeddings-practice'
/content
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git


In [7]:
import os
from google.colab import userdata

GH_TOKEN = userdata.get("GITHUB_TOKEN")
USER = "Muskan-Savita"
REPO = "gemma-embeddings-practice"

%cd /content
!rm -rf gemma-embeddings-practice
!git clone https://{GH_TOKEN}@github.com/{USER}/{REPO}.git
%cd /content/gemma-embeddings-practice
!git config --global user.email "savitamuskan1404@gmail.com"
!git config --global user.name "Muskan-Savita"
!mkdir -p app
!touch app/__init__.py

/content
Cloning into 'gemma-embeddings-practice'...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 6 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (6/6), done.
/content/gemma-embeddings-practice


In [8]:
%%writefile /content/gemma-embeddings-practice/app/detection.py
import re

RULES = {
    "CREDIT_CARD": re.compile(r"\b(?:\d[ -]?){12,18}\d\b"),
    "EMAIL": re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+"),
    "SSN":   re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
    "PHONE": re.compile(r"(?<!\d)\+?\d[\d\s-]{8,}\d(?!\d)"),
    "IPV4":  re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b"),
}

def luhn_valid(number: str) -> bool:
    digits = [int(c) for c in number if c.isdigit()]
    if not 13 <= len(digits) <= 19:
        return False
    total = 0
    for i, d in enumerate(reversed(digits)):
        if i % 2 == 1:
            d *= 2
            if d > 9:
                d -= 9
        total += d
    return total % 10 == 0

def detect(text: str) -> list[tuple[str, int, int]]:
    out = []
    for label, pat in RULES.items():
        for m in pat.finditer(text):
            if label == "CREDIT_CARD" and not luhn_valid(m.group(0)):
                continue
            out.append((label, m.start(), m.end()))

    out.sort(key=lambda x: (x[1], -(x[2] - x[1])))
    result, last_end = [], -1
    for f in out:
        if f[1] >= last_end:
            result.append(f)
            last_end = f[2]
    return result

def sanitize(text: str) -> str:
    for label, s, e in sorted(detect(text), key=lambda x: x[1], reverse=True):
        text = text[:s] + f"<{label}>" + text[e:]
    return text

Writing /content/gemma-embeddings-practice/app/detection.py


In [9]:
%cd /content/gemma-embeddings-practice
!git add app
!git commit -m "Add PII detection module"
!git push origin main

/content/gemma-embeddings-practice
[main 6559ade] Add PII detection module
 2 files changed, 43 insertions(+)
 create mode 100644 app/__init__.py
 create mode 100644 app/detection.py
Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (5/5), 1.00 KiB | 1.00 MiB/s, done.
Total 5 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/Muskan-Savita/gemma-embeddings-practice.git
   c8fa11a..6559ade  main -> main
